In [8]:
import glob
import os
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.fill import fillnodata
from scipy import ndimage as ndi

paths = sorted(glob.glob("/Users/willmcmahon/Downloads/NL/*.tiff"))
print(f"Found {len(paths)} files to merge")
srcs = [rasterio.open(p) for p in paths]
mosaic, transform = merge(srcs)
profile = srcs[0].profile.copy()
profile.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=transform)
nodata = srcs[0].nodata if srcs[0].nodata is not None else -9999
for s in srcs:
    s.close()

arr = mosaic[0]
missing = arr == nodata

# opening wipes out the small speckle and leaves only the big empty areas
border = ndi.binary_opening(missing, structure=np.ones((3, 3)), iterations=12)

filled = fillnodata(arr, mask=(~missing).astype("uint8"),
                    max_search_distance=50, smoothing_iterations=1)
filled[border] = nodata

profile.update(nodata=nodata)
os.makedirs("out", exist_ok=True)
with rasterio.open("out/mosaic_filled.tif", "w", **profile) as dst:
    dst.write(filled, 1)

Found 4 files to merge
